# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content page for one client on one reporting day. I verified this using report_date, client_hash_id, and content_hash_id; there were 0 duplicate combinations, confirming the page-day grain.

**Time window:** For this assignment, I will use the March 2026 partition (2026-03-01 to 2026-03-31) as my development and verification window. I chose a mid-panel month so I can develop and test the data contract and features without using the final June 2026 month, which should be treated as a sealed/future test period.

In [3]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset:", info.id)
print("Access confirmed.")

NameError: name 'HF_TOKEN' is not defined

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="README.md",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Warehouse connection working.")
print(path)

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    print(f)

In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_path)

print("Rows:", len(march_df))
print("Columns:", len(march_df.columns))
print("Date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

In [ ]:
print(march_df.columns.tolist())
march_df.head()

In [ ]:
duplicates = (
    march_df
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print("Duplicate page-day combinations:", len(duplicates))

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# Profile all 30 columns
column_profile = pd.DataFrame({
    "column": march_df.columns,
    "non_null_count": march_df.notna().sum().values,
    "missing_count": march_df.isna().sum().values,
    "non_null_pct": (march_df.notna().mean() * 100).round(2).values
})

column_profile = column_profile.sort_values(
    "non_null_pct",
    ascending=False
)

column_profile

In [ ]:
# Check whether metric values are actually present when the data source
# is marked unavailable

print("GSC availability vs impressions:")
print(pd.crosstab(
    march_df["gsc_data_available"],
    march_df["gsc_impressions"].notna()
))

print("\nGSC availability vs average position:")
print(pd.crosstab(
    march_df["gsc_data_available"],
    march_df["gsc_avg_position"].notna()
))

print("\nGA4 availability vs pageviews:")
print(pd.crosstab(
    march_df["ga4_data_available"],
    march_df["ga4_pageviews"].notna()
))

In [ ]:
ga4_check = march_df.groupby("ga4_data_available").agg(
    rows=("content_hash_id", "size"),
    pageviews_non_null=("ga4_pageviews", "count"),
    sessions_non_null=("ga4_sessions", "count"),
    engaged_sessions_non_null=("ga4_engaged_sessions", "count"),
    engagement_sec_non_null=("ga4_total_engagement_sec", "count")
)

ga4_check

In [ ]:
march_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "sessions_ai"
    ]
].describe()

march_df[
    [
        "sessions_organic",
        "sessions_ai"
    ]
].isna().sum()

**2. Fields: Feature / Label / Context / Excluded**

**Context**

These fields help identify, organize, filter, or interpret the data, but they will not be used by the model as predictive features.

**report_date **— identifies the date of the observation.
**client_hash_id** — identifies the client and can be used for grouping or splitting.
**content_hash_id** — identifies the page/content item.
**client_has_gsc** — indicates whether the client has GSC.
**client_has_ga4** — indicates whether the client has GA4.
**gsc_data_available** — indicates whether usable GSC data is available for the observation.
**ga4_data_available** — indicates whether usable GA4 data is available for the observation.

**Features**

**For this assignment, I will use the following five initial features:**

**gsc_impressions **— measures the page's search visibility.
**gsc_clicks** — measures the search traffic received by the page.
**gsc_avg_position** — represents the page's average search position.
**sessions_organic** — measures organic sessions reaching the page.
**sessions_ai** — measures sessions attributed to AI sources.

Together, these features provide information about search visibility, search traffic, search ranking, organic traffic, and AI-driven traffic.

The availability of these features will also be considered during preprocessing. In particular, gsc_avg_position is available for 36.69% of rows, while sessions_organic and sessions_ai are available for 69.33% of rows, so missing values will not automatically be treated as zero.

**Label / Proxy**

There is no direct label in the March performance fields that tells us whether a page should receive priority.

Therefore, I will not use a current performance metric as the label. Instead, the label/proxy will be derived from an observed future outcome window, allowing the model to use information available earlier to assess what happens to the page later.

**Excluded**
gsc_sum_position — excluded because it is closely related to gsc_avg_position and provides redundant position information.
ga4_pageviews — excluded because it overlaps with other traffic measures and is less directly aligned with our selected feature set.
ga4_sessions — excluded because it overlaps with other traffic-volume measures.
ga4_users — excluded because it provides another closely related measure of traffic volume.
ga4_engaged_sessions — excluded from the final five because we selected sessions_organic and sessions_ai as more directly relevant traffic signals for this initial frame.
ga4_total_engagement_sec — excluded for the same reason; it is useful engagement information but is not part of our selected five-feature frame.
sessions_direct — excluded because direct traffic is less directly aligned with our organic content-prioritization decision.
sessions_referral — excluded because referral traffic is less directly aligned with the current decision.
sessions_social — excluded because social traffic is less directly aligned with the current decision.
sessions_paid — excluded because paid traffic is less directly aligned with the organic content-prioritization decision.
ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — excluded because they are detailed components of AI traffic and would add unnecessary granularity and redundancy when sessions_ai already captures the broader AI-traffic signal.
scroll_events — excluded because it is a relatively crude engagement signal and is less directly useful than the selected traffic and search-performance features.

Excluded does not mean these fields are useless. It means that for this initial five-feature frame, we deliberately chose not to use them because they are redundant, less aligned with the decision, or less informative than the selected features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
total_rows = len(march_df)

gsc_available = march_df["gsc_data_available"].eq(True).sum()
ga4_available = march_df["ga4_data_available"].eq(True).sum()

print("Total March rows:", total_rows)
print("GSC available rows:", gsc_available)
print("GA4 available rows:", ga4_available)

print("GSC available %:", round(gsc_available / total_rows * 100, 2))
print("GA4 available %:", round(ga4_available / total_rows * 100, 2))

**3.1 Grain**

I verified that the combination of report_date, client_hash_id, and content_hash_id contains no duplicate records in the March 2026 data.

The result was 0 duplicate page-day combinations, confirming that one row represents one page for one client on one reporting date.

**3.2 Row Count and Date Window**

The March 2026 slice contains 9,841,378 rows, with a date range from March 1, 2026 to March 31, 2026.

Therefore, our analysis is based on the March 2026 data window.

**3.3 Data Availability**

Out of the 9,841,378 March rows:

**GSC data:** 3,611,061 rows (36.69%) available
**GA4 data:** 413,966 rows (4.21%) available



**3.4 Perform the leakage trap**

In [ ]:
features_df = march_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "sessions_ai"
    ]
].copy()

features_df.head()

In [ ]:
features_df.shape
features_df.isna().sum()

In [ ]:
from datasets import load_dataset

april = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-04/data_0.parquet",
    split="train"
)

april_df = april.to_pandas()

print(april_df.shape)
print(april_df.columns.tolist())

In [ ]:
# Check how many March page-client combinations also appear in April

march_pages = march_df[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()

april_pages = april_df[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()

matched_pages = march_pages.merge(
    april_pages,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Unique March page-client pairs:", len(march_pages))
print("Unique April page-client pairs:", len(april_pages))
print("March pairs found in April:", len(matched_pages))

In [ ]:
# Keep the March features we care about
march_compare = march_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "sessions_ai"
    ]
].copy()

# Keep the same columns from April
april_compare = april_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "sessions_organic",
        "sessions_ai"
    ]
].copy()

# Rename April columns so we can distinguish them
april_compare = april_compare.rename(columns={
    "gsc_impressions": "april_impressions",
    "gsc_clicks": "april_clicks",
    "gsc_avg_position": "april_avg_position",
    "sessions_organic": "april_sessions_organic",
    "sessions_ai": "april_sessions_ai"
})

# Match March pages with their April observations
march_april = march_compare.merge(
    april_compare,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Matched rows:", len(march_april))
march_april.head()

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**This dataset has several limitations that affect what the model can reliably tell us.**


**Uneven data availability:** GSC and GA4 data are not available for every page and date. In March 2026, GSC data is available for 36.69% of rows, while GA4 data availability is only 4.21% according to the availability flags. Therefore, missing data cannot automatically be interpreted as zero activity.

**GSC-only history:** Some earlier observations may contain GSC information without corresponding GA4 information. This means we cannot assume that every page has the same history or the same set of measurable signals.

**Different observation windows:** The available fields may represent different reporting or collection windows. We therefore cannot assume that every column describes exactly the same period without checking its definition.

**Future outcomes are not directly observed in the current row:** The March performance data tells us what was known at that time, but it does not directly tell us whether a page will decline or require attention later. A future outcome must be derived from a later observation window.

**No causal explanation:**
The data can show patterns and associations between page characteristics and later outcomes, but it cannot prove that one feature directly caused a page's performance to change.

**Limited generalization:** The model will learn from the historical pages and clients represented in this warehouse. Its predictions may not perform equally well for new clients, pages, or situations that are very different from the historical data.

**Overall limitation:** This dataset can help us identify and rank pages based on observed patterns, but it cannot guarantee why a page changed, what caused the change, or what will happen in every future situation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.